# Nettoyage des données — Appendicite pédiatrique

**Objectif** : Préparer la base brute `app_data.xlsx` pour l'analyse statistique et le machine learning.  
**Sortie** : `data/raw/app_data_sans_critiques.xlsx` — base nettoyée, sans les colonnes critiques 

---

### Étapes du pipeline
1. Chargement et exploration initiale
2. Suppression des doublons
3. Gestion des valeurs manquantes
   - Variables numériques → imputation par la médiane
   - Variables catégorielles → imputation par le mode
   - Variables critiques (taux de manquants élevé) → **suppression des colonnes**
4. Détection et traitement des valeurs aberrantes
   - Détection par IQR
   - Détection par Z-score
   - Suppression des valeurs biologiquement impossibles
   - Winsorisation
   - Log-transformation de la CRP
5. Export de la base nettoyée

---
## 1. Chargement des packages et de la base

In [17]:
import os
import numpy as np
import pandas as pd
from scipy import stats

# Chargement de la base brute
df = pd.read_excel("../data/raw/dataset.xlsx")

print(f"Dimensions initiales : {df.shape[0]} lignes × {df.shape[1]} colonnes")
df.head()

Dimensions initiales : 782 lignes × 58 colonnes


,Age,BMI,Sex,Height,Weight,Length_of_Stay,Management,Severity,Diagnosis_Presumptive,Diagnosis,...,Abscess_Location,Pathological_Lymph_Nodes,Lymph_Nodes_Location,Bowel_Wall_Thickening,Conglomerate_of_Bowel_Loops,Ileus,Coprostasis,Meteorism,Enteritis,Gynecological_Findings
0,12.68,16.9,female,148.0,37.0,3.0,conservative,uncomplicated,appendicitis,appendicitis,...,NaN,yes,reUB,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,14.10,31.9,male,147.0,69.5,2.0,conservative,uncomplicated,appendicitis,no appendicitis,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yes,NaN,NaN
2,14.14,23.3,female,163.0,62.0,4.0,conservative,uncomplicated,appendicitis,no appendicitis,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yes,yes,NaN
3,16.37,20.6,female,165.0,56.0,3.0,conservative,uncomplicated,appendicitis,no appendicitis,...,NaN,yes,reUB,NaN,NaN,NaN,NaN,NaN,yes,NaN
4,11.08,16.9,female,163.0,45.0,3.0,conservative,uncomplicated,appendicitis,appendicitis,...,NaN,yes,reUB,NaN,NaN,NaN,NaN,NaN,yes,NaN


In [18]:
# Aperçu des types de colonnes et des valeurs non-nulles
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 782 entries, 0 to 781
Data columns (total 58 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Age                               781 non-null    float64
 1   BMI                               755 non-null    float64
 2   Sex                               780 non-null    str    
 3   Height                            756 non-null    float64
 4   Weight                            779 non-null    float64
 5   Length_of_Stay                    778 non-null    float64
 6   Management                        781 non-null    str    
 7   Severity                          781 non-null    str    
 8   Diagnosis_Presumptive             780 non-null    str    
 9   Diagnosis                         780 non-null    str    
 10  Alvarado_Score                    730 non-null    float64
 11  Paedriatic_Appendicitis_Score     730 non-null    float64
 12  Appendix_on_US     

In [19]:
# Statistiques descriptives des variables numériques
df.describe()

,Age,BMI,Height,Weight,Length_of_Stay,Alvarado_Score,Paedriatic_Appendicitis_Score,Appendix_Diameter,Body_Temperature,WBC_Count,Neutrophil_Percentage,Segmented_Neutrophils,RBC_Count,Hemoglobin,RDW,Thrombocyte_Count,CRP,US_Number
count,781.000000,755.000000,756.000000,779.000000,778.000000,730.000000,730.000000,498.000000,775.000000,776.000000,679.000000,54.000000,764.000000,764.000000,756.000000,764.000000,771.000000,760.000000
mean,11.346483,18.906916,148.017460,43.172542,4.284062,5.921918,5.253425,7.762651,37.404516,12.670683,71.791163,64.929630,4.799490,13.380497,13.180291,285.252618,31.387899,425.515789
std,3.529979,4.385252,19.732016,17.390984,2.574057,2.155972,1.958456,2.536671,0.903678,5.366525,14.463656,15.085025,0.499012,1.393271,4.538774,72.494373,57.433588,271.585211
min,0.000000,7.827983,53.000000,3.960000,1.000000,0.000000,0.000000,2.700000,26.900000,2.600000,27.200000,32.000000,3.620000,8.200000,11.200000,91.000000,0.000000,1.000000
25%,9.200000,15.725294,137.000000,29.500000,3.000000,4.000000,4.000000,6.000000,36.800000,8.200000,61.400000,54.500000,4.537500,12.600000,12.300000,236.000000,1.000000,198.750000
50%,11.438741,18.062284,149.650000,41.400000,3.000000,6.000000,5.000000,7.500000,37.200000,12.000000,75.500000,64.500000,4.780000,13.300000,12.700000,276.000000,7.000000,398.500000
75%,14.099932,21.179011,163.000000,54.000000,5.000000,8.000000,7.000000,9.100000,37.900000,16.200000,83.600000,77.500000,5.020000,14.000000,13.300000,330.000000,33.000000,613.250000
max,18.360000,38.156221,192.000000,103.000000,28.000000,10.000000,10.000000,17.000000,40.200000,37.700000,97.700000,91.000000,14.000000,36.000000,86.900000,708.000000,365.000000,992.000000


---
## 2. Suppression des doublons

In [20]:
n_doublons = df.duplicated().sum()
print(f"Nombre de lignes dupliquées : {n_doublons}")

# Suppression si des doublons existent
if n_doublons > 0:
    df = df.drop_duplicates()
    print(f"→ Base après suppression des doublons : {df.shape[0]} lignes")

Nombre de lignes dupliquées : 0


---
## 3. Gestion des valeurs manquantes

In [21]:
# Vue d'ensemble des valeurs manquantes par colonne
manquants = df.isnull().sum()
manquants_pct = (manquants / len(df) * 100).round(1)

resume_manquants = pd.DataFrame({
    'Valeurs manquantes': manquants,
    'Pourcentage (%)': manquants_pct
}).query('`Valeurs manquantes` > 0').sort_values('Pourcentage (%)', ascending=False)

print(resume_manquants)

                                  Valeurs manquantes  Pourcentage (%)
Abscess_Location                                 769             98.3
Gynecological_Findings                           756             96.7
Conglomerate_of_Bowel_Loops                      739             94.5
Segmented_Neutrophils                            728             93.1
Ileus                                            722             92.3
Perfusion                                        719             91.9
Enteritis                                        716             91.6
Appendicolith                                    713             91.2
Coprostasis                                      711             90.9
Perforation                                      701             89.6
Appendicular_Abscess                             697             89.1
Bowel_Wall_Thickening                            683             87.3
Lymph_Nodes_Location                             661             84.5
Target_Sign         

### 3a. Variables numériques → imputation par la médiane

La médiane est préférée à la moyenne car elle est robuste aux valeurs extrêmes,  
fréquentes dans les données biologiques (CRP, leucocytes, etc.).

In [22]:
colonnes_numeriques_a_imputer = [
    'Appendix_Diameter', 'Neutrophil_Percentage', 'Alvarado_Score',
    'Paedriatic_Appendicitis_Score', 'BMI', 'Height', 'RDW',
    'US_Number', 'Hemoglobin', 'Thrombocyte_Count', 'RBC_Count',
    'CRP', 'Body_Temperature', 'WBC_Count', 'Length_of_Stay',
    'Weight', 'Age'
]

for col in colonnes_numeriques_a_imputer:
    if col in df.columns:
        mediane = df[col].median()
        nb_imputes = df[col].isnull().sum()
        df[col] = df[col].fillna(mediane)
        if nb_imputes > 0:
            print(f"{col} : {nb_imputes} valeur(s) imputée(s) avec la médiane ({mediane:.2f})")

print("\n✓ Imputation des variables numériques terminée.")

Appendix_Diameter : 284 valeur(s) imputée(s) avec la médiane (7.50)
Neutrophil_Percentage : 103 valeur(s) imputée(s) avec la médiane (75.50)
Alvarado_Score : 52 valeur(s) imputée(s) avec la médiane (6.00)
Paedriatic_Appendicitis_Score : 52 valeur(s) imputée(s) avec la médiane (5.00)
BMI : 27 valeur(s) imputée(s) avec la médiane (18.06)
Height : 26 valeur(s) imputée(s) avec la médiane (149.65)
RDW : 26 valeur(s) imputée(s) avec la médiane (12.70)
US_Number : 22 valeur(s) imputée(s) avec la médiane (398.50)
Hemoglobin : 18 valeur(s) imputée(s) avec la médiane (13.30)
Thrombocyte_Count : 18 valeur(s) imputée(s) avec la médiane (276.00)
RBC_Count : 18 valeur(s) imputée(s) avec la médiane (4.78)
CRP : 11 valeur(s) imputée(s) avec la médiane (7.00)
Body_Temperature : 7 valeur(s) imputée(s) avec la médiane (37.20)
WBC_Count : 6 valeur(s) imputée(s) avec la médiane (12.00)
Length_of_Stay : 4 valeur(s) imputée(s) avec la médiane (3.00)
Weight : 3 valeur(s) imputée(s) avec la médiane (41.40)
Age

### 3b. Variables catégorielles → imputation par le mode

Pour les variables binaires ou nominales, on remplace les valeurs manquantes  
par la catégorie la plus fréquente (mode).

In [23]:
colonnes_cat = [
    'RBC_in_Urine', 'Ketones_in_Urine', 'WBC_in_Urine',
    'Ipsilateral_Rebound_Tenderness', 'Sex', 'Diagnosis', 'Severity',
    'Management', 'Neutrophilia', 'Migratory_Pain', 'Lower_Right_Abd_Pain',
    'Contralateral_Rebound_Tenderness', 'Coughing_Pain', 'Nausea',
    'Loss_of_Appetite', 'Dysuria', 'Stool', 'Peritonitis', 'Psoas_Sign',
    'Appendix_on_US', 'US_Performed', 'Free_Fluids', 'Diagnosis_Presumptive'
]

for col in colonnes_cat:
    if col in df.columns:
        mode_val = df[col].mode()[0]
        nb_imputes = df[col].isnull().sum()
        df[col] = df[col].fillna(mode_val)
        if nb_imputes > 0:
            print(f"{col} : {nb_imputes} valeur(s) imputée(s) avec le mode ('{mode_val}')")

print("\n✓ Imputation des variables catégorielles terminée.")

RBC_in_Urine : 206 valeur(s) imputée(s) avec le mode ('no')
Ketones_in_Urine : 200 valeur(s) imputée(s) avec le mode ('no')
WBC_in_Urine : 199 valeur(s) imputée(s) avec le mode ('no')
Ipsilateral_Rebound_Tenderness : 163 valeur(s) imputée(s) avec le mode ('no')
Sex : 2 valeur(s) imputée(s) avec le mode ('male')
Diagnosis : 2 valeur(s) imputée(s) avec le mode ('appendicitis')
Severity : 1 valeur(s) imputée(s) avec le mode ('uncomplicated')
Management : 1 valeur(s) imputée(s) avec le mode ('conservative')
Neutrophilia : 50 valeur(s) imputée(s) avec le mode ('no')
Migratory_Pain : 9 valeur(s) imputée(s) avec le mode ('no')
Lower_Right_Abd_Pain : 8 valeur(s) imputée(s) avec le mode ('yes')
Contralateral_Rebound_Tenderness : 15 valeur(s) imputée(s) avec le mode ('no')
Coughing_Pain : 16 valeur(s) imputée(s) avec le mode ('no')
Nausea : 8 valeur(s) imputée(s) avec le mode ('yes')
Loss_of_Appetite : 10 valeur(s) imputée(s) avec le mode ('yes')
Dysuria : 29 valeur(s) imputée(s) avec le mode ('

### 3c. Variables critiques → suppression des colonnes

Ces colonnes présentent un taux de valeurs manquantes trop élevé pour permettre  
une imputation fiable. Elles sont **supprimées** de la base exportée.  
*(Stratégie alternative possible : les conserver sous forme d'indicateurs binaires 0/1,  
mais ce choix dépend de l'usage analytique prévu.)*

In [24]:
colonnes_critiques = [
    'Abscess_Location', 'Gynecological_Findings', 'Conglomerate_of_Bowel_Loops',
    'Segmented_Neutrophils', 'Ileus', 'Perfusion', 'Enteritis', 'Appendicolith',
    'Coprostasis', 'Perforation', 'Appendicular_Abscess', 'Bowel_Wall_Thickening',
    'Lymph_Nodes_Location', 'Target_Sign', 'Meteorism', 'Pathological_Lymph_Nodes',
    'Appendix_Wall_Layers', 'Surrounding_Tissue_Reaction'
]

# Ne supprimer que les colonnes effectivement présentes dans le DataFrame
colonnes_a_supprimer = [col for col in colonnes_critiques if col in df.columns]

df = df.drop(columns=colonnes_a_supprimer)

print(f"Colonnes critiques supprimées ({len(colonnes_a_supprimer)}) :")
for col in colonnes_a_supprimer:
    print(f"  - {col}")

print(f"\nDimensions après suppression : {df.shape[0]} lignes × {df.shape[1]} colonnes")

Colonnes critiques supprimées (18) :
  - Abscess_Location
  - Gynecological_Findings
  - Conglomerate_of_Bowel_Loops
  - Segmented_Neutrophils
  - Ileus
  - Perfusion
  - Enteritis
  - Appendicolith
  - Coprostasis
  - Perforation
  - Appendicular_Abscess
  - Bowel_Wall_Thickening
  - Lymph_Nodes_Location
  - Target_Sign
  - Meteorism
  - Pathological_Lymph_Nodes
  - Appendix_Wall_Layers
  - Surrounding_Tissue_Reaction

Dimensions après suppression : 782 lignes × 40 colonnes


---
## 4. Détection et traitement des valeurs aberrantes

### 4a. Détection par la méthode IQR

Un point est considéré aberrant s'il est en dehors de l'intervalle  
`[Q1 − 1.5×IQR, Q3 + 1.5×IQR]` (règle de Tukey).

In [25]:
colonnes_numeriques = [
    'Age', 'BMI', 'CRP', 'WBC_Count', 'Hemoglobin',
    'Body_Temperature', 'RDW', 'Appendix_Diameter',
    'Alvarado_Score', 'Paedriatic_Appendicitis_Score',
    'Length_of_Stay', 'Weight', 'Height', 'RBC_Count',
    'Thrombocyte_Count', 'Neutrophil_Percentage'
]

print("=== Détection IQR ===")
for col in colonnes_numeriques:
    if col not in df.columns:
        continue
    s = df[col].dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    bb, bh = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n = ((s < bb) | (s > bh)).sum()
    print(f"{col:40s} : {n:3d} outliers | bornes = [{bb:.2f}, {bh:.2f}]")

=== Détection IQR ===
Age                                      :   5 outliers | bornes = [1.90, 21.39]
BMI                                      :  30 outliers | bornes = [7.99, 28.83]
CRP                                      :  91 outliers | bornes = [-45.50, 78.50]
WBC_Count                                :   7 outliers | bornes = [-3.55, 28.05]
Hemoglobin                               :  26 outliers | bornes = [10.75, 15.95]
Body_Temperature                         :  12 outliers | bornes = [35.15, 39.55]
RDW                                      :  23 outliers | bornes = [10.80, 14.80]
Appendix_Diameter                        : 217 outliers | bornes = [5.50, 9.50]
Alvarado_Score                           :   0 outliers | bornes = [-2.00, 14.00]
Paedriatic_Appendicitis_Score            :  14 outliers | bornes = [1.00, 9.00]
Length_of_Stay                           :  44 outliers | bornes = [0.00, 8.00]
Weight                                   :   8 outliers | bornes = [-7.25, 90.75]
H

### 4b. Détection par Z-score

Un Z-score > 3 signifie que la valeur s'écarte de plus de 3 écarts-types de la moyenne,  
ce qui correspond à une probabilité < 0.3% sous une loi normale.

In [26]:
print("=== Détection par Z-score (seuil |z| > 3) ===")
for col in colonnes_numeriques:
    if col not in df.columns:
        continue
    s = df[col].dropna()
    z = np.abs(stats.zscore(s))
    print(f"{col:40s} : {(z > 3).sum():3d} outliers")

=== Détection par Z-score (seuil |z| > 3) ===
Age                                      :   3 outliers
BMI                                      :  12 outliers
CRP                                      :  23 outliers
WBC_Count                                :   6 outliers
Hemoglobin                               :   2 outliers
Body_Temperature                         :   2 outliers
RDW                                      :   3 outliers
Appendix_Diameter                        :  10 outliers
Alvarado_Score                           :   0 outliers
Paedriatic_Appendicitis_Score            :   0 outliers
Length_of_Stay                           :  15 outliers
Weight                                   :   4 outliers
Height                                   :   3 outliers
RBC_Count                                :   2 outliers
Thrombocyte_Count                        :   5 outliers
Neutrophil_Percentage                    :   3 outliers


### 4c. Suppression des valeurs biologiquement impossibles

Certaines valeurs sont physiologiquement impossibles et signalent des erreurs de saisie.  
Elles sont supprimées directement (filtrage par seuils cliniques).

In [27]:
n_avant = len(df)

df = df[df['Body_Temperature'] > 34]   # Hypothermie sévère non compatible avec les soins
df = df[df['Hemoglobin'] <= 20]        # Hémoglobine > 20 g/dL : impossible
df = df[df['RDW'] <= 30]              # RDW > 30% : impossible
df = df[df['RBC_Count'] <= 8]         # GR > 8 M/µL : impossible

n_apres = len(df)
print(f"Lignes supprimées (valeurs biologiquement impossibles) : {n_avant - n_apres}")
print(f"Taille de la base restante : {n_apres} lignes")

Lignes supprimées (valeurs biologiquement impossibles) : 6
Taille de la base restante : 776 lignes


### 4d. Winsorisation

Pour les variables retenues mais présentant des valeurs extrêmes plausibles,  
on applique une winsorisation (plafonnement aux bornes IQR) plutôt qu'une suppression.  
Cela préserve toutes les observations tout en limitant l'influence des extrêmes.

In [28]:
colonnes_winsor = ['BMI', 'WBC_Count', 'Length_of_Stay',
                   'Thrombocyte_Count', 'Appendix_Diameter']

for col in colonnes_winsor:
    if col not in df.columns:
        continue
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    borne_basse = Q1 - 1.5 * IQR
    borne_haute = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=borne_basse, upper=borne_haute)
    print(f"{col:30s} : winsorisé à [{borne_basse:.2f}, {borne_haute:.2f}]")

print("\n✓ Winsorisation terminée.")

BMI                            : winsorisé à [7.96, 28.86]
WBC_Count                      : winsorisé à [-3.61, 28.09]
Length_of_Stay                 : winsorisé à [0.00, 8.00]
Thrombocyte_Count              : winsorisé à [97.62, 466.62]
Appendix_Diameter              : winsorisé à [5.50, 9.50]

✓ Winsorisation terminée.


### 4e. Log-transformation de la CRP

La CRP (Protéine C-Réactive) suit typiquement une distribution très asymétrique à droite  
dans les données cliniques. La transformation `log(1 + CRP)` permet de la normaliser  
et d'améliorer les performances des modèles statistiques.  
Le `+1` évite les problèmes liés aux valeurs nulles (`log(0)` est indéfini).

In [29]:
df['CRP_log'] = np.log1p(df['CRP'])

print(f"CRP originale — min: {df['CRP'].min():.1f}, max: {df['CRP'].max():.1f}, médiane: {df['CRP'].median():.1f}")
print(f"CRP_log      — min: {df['CRP_log'].min():.2f}, max: {df['CRP_log'].max():.2f}, médiane: {df['CRP_log'].median():.2f}")

CRP originale — min: 0.0, max: 365.0, médiane: 7.0
CRP_log      — min: 0.00, max: 5.90, médiane: 2.08


---
## 5. Vérification finale avant export


In [ ]:
# ------------------3. Vérification des valeurs manquantes et doublons---------------------------
print("valeur manquante :", df.isnull().sum())
print("Nombre de lignes en double :", df.duplicated().sum())
total_missing = df.isnull().sum().sum()
total_duplicates = df.duplicated().sum()
if total_missing == 0 and total_duplicates == 0:
    print("Le fichier est bien nettoyé : aucune valeur manquante ni doublon.")
else:
    print(f"Le fichier n'est pas totalement nettoyé : {total_missing} valeurs manquantes, {total_duplicates} doublons.")

valeur manquante : Age                                 0
BMI                                 0
Sex                                 0
Height                              0
Weight                              0
Length_of_Stay                      0
Management                          0
Severity                            0
Diagnosis_Presumptive               0
Diagnosis                           0
Alvarado_Score                      0
Paedriatic_Appendicitis_Score       0
Appendix_on_US                      0
Appendix_Diameter                   0
Migratory_Pain                      0
Lower_Right_Abd_Pain                0
Contralateral_Rebound_Tenderness    0
Coughing_Pain                       0
Nausea                              0
Loss_of_Appetite                    0
Body_Temperature                    0
WBC_Count                           0
Neutrophil_Percentage               0
Neutrophilia                        0
RBC_Count                           0
Hemoglobin                     

---
## 6. Export de la base nettoyée

In [31]:
# Créer le dossier de sortie s'il n'existe pas encore
os.makedirs("../data/processed", exist_ok=True)

# Export uniquement si la base est propre
if total_missing == 0:
    chemin_sortie = "../data/processed/app_data_sans_critiques.xlsx"
    df.to_excel(chemin_sortie, index=False)
    print(f"✓ Base exportée avec succès : {chemin_sortie}")
    print(f"  Dimensions finales : {df.shape[0]} lignes × {df.shape[1]} colonnes")
else:
    print(f"⚠ Export annulé : {total_missing} valeurs manquantes détectées. Corriger avant d'exporter.")


✓ Base exportée avec succès : ../data/processed/app_data_sans_critiques.xlsx
  Dimensions finales : 776 lignes × 41 colonnes


---
## Récapitulatif du pipeline

| Étape | Action | Résultat |
|---|---|---|
| Doublons | Suppression | Lignes dupliquées retirées |
| Manquants numériques | Imputation médiane | 17 variables traitées |
| Manquants catégoriels | Imputation mode | 23 variables traitées |
| Colonnes critiques | **Suppression** | 18 colonnes supprimées |
| Valeurs impossibles | Suppression de lignes | Filtres biologiques appliqués |
| Valeurs extrêmes | Winsorisation | 5 variables plafonnées |
| CRP | Log-transformation | Colonne `CRP_log` ajoutée |